In [1]:
!pip install nibabel

In [31]:
import nibabel as nib
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, IntSlider

def load_nifti(filepath):
    """
    Load a NIfTI file and return the image object and data as numpy array.
    """
    img = nib.load(filepath)
    data = img.get_fdata()
    return img, data

def show_metadata(img):
    """
    Print metadata of the NIfTI file.
    """
    print("Shape (x, y, z):", img.shape)
    print("Voxel size:", img.header.get_zooms())
    print("Affine:\n", img.affine)

def visualize_overlay(img_data, mask_data, axis=2, alpha=0.4, mask_color="Greens"):
    """
    Interactive viewer to scroll through slices of MRI with segmentation overlay.

    Parameters:
        img_data : numpy array (MRI volume)
        mask_data : numpy array (segmentation mask, same shape as img_data)
        axis : int → 0 (sagittal), 1 (coronal), 2 (axial)
        alpha : float → transparency for the mask overlay
        mask_color : str → matplotlib colormap (e.g., 'Reds', 'Greens')
    """
    assert img_data.shape == mask_data.shape, "MRI and mask must have the same shape"

    def plot_slice(slice_idx):
        # Extract slices along given axis
        base_slice = np.rot90(img_data.take(slice_idx, axis=axis))
        mask_slice = np.rot90(mask_data.take(slice_idx, axis=axis))

        plt.figure(figsize=(6,6))
        plt.imshow(base_slice, cmap="gray")
        plt.imshow(mask_slice, cmap=mask_color, alpha=alpha)
        plt.title(f"Axis {axis} | Slice {slice_idx}")
        plt.axis("off")
        plt.show()

    interact(
        plot_slice,
        slice_idx=IntSlider(min=0, max=img_data.shape[axis]-1, step=1, value=img_data.shape[axis]//2)
    )


In [34]:

_, img_data = load_nifti("/content/21 t1_tirm_tra_dark-fluid_fs.nii")
_, mask_data = load_nifti("/content/Segmentation-Segment_1-label.nii")

visualize_overlay(img_data, mask_data, axis=2, alpha=0.4, mask_color="Blues")

interactive(children=(IntSlider(value=13, description='slice_idx', max=26), Output()), _dom_classes=('widget-i…